# 03 - 线性回归实战

## 学习目标
- 理解线性回归的数学原理
- 使用 scikit-learn 实现线性回归
- 掌握完整的 ML 工作流：数据 → 预处理 → 训练 → 评估 → 可视化

## 数据集
加州房价数据集 (California Housing)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. 数据加载与探索

In [ ]:
# 加载数据
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

print(f'数据集形状: {X.shape}')
print(f'特征列: {list(X.columns)}')
print(f'\n目标变量 (房价中位数, 单位: 十万美元):')
print(pd.Series(y).describe())

In [ ]:
# 数据概览
X.head()

In [ ]:
# 特征分布
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(X.columns):
    axes[i].hist(X[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

plt.suptitle('特征分布', fontsize=14)
plt.tight_layout()
plt.show()

## 2. 数据预处理

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'训练集: {X_train.shape}')
print(f'测试集: {X_test.shape}')

# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # 注意：只用 transform！

print(f'\n标准化后训练集均值: {X_train_scaled.mean(axis=0).round(4)}')
print(f'标准化后训练集标准差: {X_train_scaled.std(axis=0).round(4)}')

## 3. 模型训练

In [ ]:
# 创建并训练线性回归模型
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# 查看模型参数
print('模型系数 (标准化后):')
print('-' * 40)
for feature, coef in sorted(zip(california.feature_names, model.coef_), key=lambda x: abs(x[1]), reverse=True):
    bar = '█' * int(abs(coef) * 5)
    print(f'  {feature:15s}: {coef:+.4f}  {bar}')
print(f'  {"截距":15s}: {model.intercept_:+.4f}')

## 4. 模型预测与评估

In [ ]:
# 预测
y_pred = model.predict(X_test_scaled)

# 计算评估指标
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('===== 模型评估结果 =====')
print(f'MSE  (均方误差):      {mse:.4f}')
print(f'RMSE (均方根误差):    {rmse:.4f}')
print(f'MAE  (平均绝对误差):  {mae:.4f}')
print(f'R²   (决定系数):      {r2:.4f}')
print(f'\n解读: R²={r2:.4f} 表示模型解释了 {r2*100:.1f}% 的目标变量方差')

## 5. 可视化分析

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 预测值 vs 真实值
axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, c='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='完美预测')
axes[0].set_xlabel('真实值')
axes[0].set_ylabel('预测值')
axes[0].set_title('预测值 vs 真实值')
axes[0].legend()

# 残差分布
residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_xlabel('残差')
axes[1].set_ylabel('频数')
axes[1].set_title('残差分布')

# 特征重要性
importance = np.abs(model.coef_)
sorted_idx = np.argsort(importance)
axes[2].barh(range(len(sorted_idx)), importance[sorted_idx], color='steelblue')
axes[2].set_yticks(range(len(sorted_idx)))
axes[2].set_yticklabels(np.array(california.feature_names)[sorted_idx])
axes[2].set_xlabel('系数绝对值')
axes[2].set_title('特征重要性 (|系数|)')

plt.tight_layout()
plt.show()

## 6. 模型比较：Ridge 与 Lasso

In [ ]:
# 比较不同正则化方法
models = {
    'LinearRegression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.01)': Lasso(alpha=0.01),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results.append({
        'Model': name,
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred),
        'R²': r2_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).set_index('Model')
print('模型比较:')
print(results_df.round(4))

## 7. 保存模型

In [ ]:
import joblib
from pathlib import Path

# 创建保存目录
save_dir = Path('../../models_saved/sklearn')
save_dir.mkdir(parents=True, exist_ok=True)

# 保存模型和 scaler
final_model = LinearRegression()
final_model.fit(X_train_scaled, y_train)

joblib.dump(final_model, save_dir / 'linear_regression.pkl')
joblib.dump(scaler, save_dir / 'scaler.pkl')

print(f'✓ 模型已保存到 {save_dir}')

## 练习

1. 尝试不同的 `test_size`，观察评估指标的变化
2. 尝试不使用 StandardScaler，对比结果
3. 调节 Ridge 和 Lasso 的 alpha 参数，观察系数变化
4. 尝试 ElasticNet（结合 L1 和 L2 正则化）

## 小结

- 线性回归是最基础的回归算法
- 特征标准化对模型性能很重要
- R² 衡量模型解释方差的能力
- Ridge/Lasso 通过正则化防止过拟合

---
下一个 Notebook: [04_logistic_regression.ipynb](04_logistic_regression.ipynb)